In [16]:
import pandas as pd
import re
import ast

# Training completions

In [3]:
data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/stage_wise/stage_1.csv") 
# data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_np.csv")

In [90]:
df = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/logs_using_subset/train_completions/qwen25_3B_drgrpo_gp8_all_train_sub_no_KL.csv")

# df1 = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/logs_using_subset/train_completions/qwen25_3B_drgrpo_gp8_all_train_sub.csv")

/scratch/ipykernel_1960876/123702552.py:1: DtypeWarning: Columns (0,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/logs_using_subset/train_completions/qwen25_3B_drgrpo_gp8_all_train_sub_no_KL.csv")


In [91]:
repeat = []
for i, id_ in enumerate(df.ID.unique()):
    if len(set(df[df['ID'] == id_]['completion']).intersection(set(df1[df1['ID'] == id_]['completion']))) == 8:
       repeat += [i]

In [92]:
repeat

[0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16]

In [93]:
df = df[df['step'] != 'step']
df['step'] = df['step'].astype(int)
# df['hybrid_reward'] = df['hybrid_rewar/d'].astype(float)
df['correctness_reward'] = df['correctness_reward'].astype(float)
# df['format_reward'] = df['format_reward'].astype(float)
# df['tag_count_reward'] = df['tag_count_reward'].astype(float)
df['advantage'] = df['advantage'].astype(float)
df = df.drop_duplicates(subset=["prompt", "completion"]).reset_index(drop=True)

In [94]:
print(len(df['ID'].unique()))
print(len(df['prompt'].unique()))

2456
2456


In [95]:
df.head()

,step,ID,ground_truth,ground_truth_text,prompt,completion,correctness_reward,advantage
0,1,NACC243361,B,Not applicable (no diagnosis of MCI),<|im_start|>system\nPlease reason step by step...,Step-by-step analysis:\n\n1. **Patient Informa...,0.0,-0.125
1,1,NACC243361,B,Not applicable (no diagnosis of MCI),<|im_start|>system\nPlease reason step by step...,Let's break down and analyze the provided pati...,0.0,-0.125
2,1,NACC243361,B,Not applicable (no diagnosis of MCI),<|im_start|>system\nPlease reason step by step...,Let's start by evaluating the details given ab...,0.0,-0.125
3,1,NACC243361,B,Not applicable (no diagnosis of MCI),<|im_start|>system\nPlease reason step by step...,"To determine the MCI subtype for the patient, ...",1.0,0.875
4,1,NACC243361,B,Not applicable (no diagnosis of MCI),<|im_start|>system\nPlease reason step by step...,Let's break down the patient's symptoms and as...,0.0,-0.125


In [96]:
df['ID'].value_counts()

ID
NACC486614    8
NACC243361    8
NACC297356    8
NACC963866    8
NACC090170    8
             ..
NACC382562    8
NACC836154    8
NACC924203    8
NACC763129    8
NACC308801    8
Name: count, Length: 2456, dtype: int64

In [97]:
cnt = []
max_cnt = 0
for i, row in df.iterrows():
    all_matches = re.findall(r'\\boxed{([A-Z])(?:\.\s*[^}]*)?}', row['completion'])
    if len(all_matches) > 1:
        max_cnt = max(max_cnt, len(all_matches))
        cnt += [(i, row['completion'])]
        
max_cnt

4

In [98]:
len(cnt)

69

In [38]:
# print(df.iloc[cnt[-1][0]]['prompt'])
# print(df.iloc[cnt[-1][0]]['ground_truth'], df.iloc[cnt[-1][0]]['ground_truth_text'])
# print(cnt[-1][1])
# print(df.iloc[cnt[-1][0]]['correctness_reward'])

In [39]:
# df['ID'].unique()

In [99]:
print(df.iloc[-1]['completion'])

To determine the appropriate mild cognitive impairment (MCI) subtype, let's break down and analyze the different cognitive and functional impairments presented in the patient's data.

1. **Cognitive Impairments:**
   - The patient exhibited agitation or aggression, depression or dysphoria, and apathy or indifference. These symptoms, rated according to the Neuropsychiatric Inventory Questionnaire (NPI-Q), indicate more serious mental health issues (agitation/ aggression, depression, respectively) rather than fundamental cognitive impairments that are hallmark of MCI.
   - Nighttime behaviors were reported as mild. No delusions, hallucinations, disinhibition, motor disturbances, or appetite and eating problems were observed.

2. **Functional Impairments:**
   - The report indicates that the patient performs tasks independently although related to memory he is struggling with. The patient identifies his difficulties with: remembering appointments, family occasions, medications, and travel

In [100]:
len(df)

19648

In [101]:
# df['reward'] = df['hybrid_reward'] + df['format_reward'] + df['tag_count_reward']
# df['reward'] = df['correctness_reward'] + df['format_reward'] + df['tag_count_reward']
df['reward'] = df['correctness_reward']

In [102]:
completion_df = pd.DataFrame(df['completion'].value_counts()).reset_index()

In [103]:
completion_df[completion_df['count'] > 1]

,completion,count


In [104]:
# print(completion_df[completion_df['count'] > 1].iloc[-1]['completion'])

In [105]:
len(df['ID'].unique())

2456

In [106]:
df['advantage_std'] = df.groupby('ID')['advantage'].transform('std')
len(df[df['advantage_std'] == 0.0].groupby('ID')['advantage'].value_counts().reset_index(name='count'))

1292

In [107]:
# df[df['advantage'] == 0].groupby('prompt')['correctness_reward'].value_counts().reset_index(name='count')['correctness_reward'].value_counts()
# sub_df = df[df['reward_std'] == 0.0].groupby('prompt')['reward'].value_counts().reset_index(name='count')
df['reward_std'] = df.groupby('ID')['reward'].transform('std')
sub_df = df[df['reward_std'] == 0.0].groupby('ID')['reward'].value_counts().reset_index(name='count')
print(len(sub_df))
sub_df['reward'].value_counts()

1292


reward
1.0    973
0.0    319
Name: count, dtype: int64

In [86]:
# check reflection patterns
cnt = []
for i, row in df.iterrows():
    if row['completion'].lower().count("differential") >= 1:
        # print("The word 'wait' appears at least twice.", i)
        # print(row['completion'])
        # break
        cnt.append(i)
        
len(cnt)

245

In [87]:
print(df.iloc[cnt[-1]]['ground_truth'])
print(df.iloc[cnt[-1]]['completion'])

A
To correctly identify the primary etiology of the patient's cognitive impairment, we need to critically review all the details provided and how they contribute to or support each potential diagnosis.

### Detailed Review of the Patient's Presentation

1. **Family History:** Reports of at least one first-degree relative (mother) with cognitive impairment suggests the patient may be at a higher genetic risk for Alzheimer's disease (AD).

2. **Medications:** The patient is taking donepezil, which is FDA-approved for the treatment of Alzheimer's disease. Additionally, she has been taking medications such as clonazepam and mirtazapine, both of which if overused could potentially contribute to or exacerbate cognitive impairment, even though they’re not the main therapies for cognitive conditions.

3. **Parkinson’s Disease:** The patient was diagnosed with a parkinsonian disorder in 2007. This supports the possibility of PD and may coexist with other conditions. However, there was no mentio

# Inference

In [181]:
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_mci/2025-06-07T195120_841fe23217cc449d/test_mci_output.jsonl"
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_mci/2025-06-07T200518_ab1e47eb4d6e4207/test_mci_output.jsonl"
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_np/2025-05-26T013154_fadcfac71d954c1b/test_np_output.jsonl"
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_np/2025-06-01T154608_8acf8fe2d7234c55/test_np_output.jsonl"
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_np/2025-05-22T102606_e437f90d9d214379/test_np_output.jsonl"
# df = pd.read_json(result_path, lines=True)

In [93]:
df = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/llm_answer_extractor/extracted_results_4/PRION/qwen25_3B_sft_PRION.csv")

In [94]:
len(df)

1780

In [95]:
sub_df = df[df['extraction_type'] == "llm"]

In [96]:
len(sub_df)

1

In [97]:
df['prediction'].value_counts()

prediction
B    1300
A     480
Name: count, dtype: int64

In [98]:
index = 10
print(df.iloc[index]['ground_truth'])
print(df.iloc[index]['prediction'])
print(df.iloc[index]['generated_text'])
print("visit_summary: ")
print(ast.literal_eval(df.iloc[index]['problem'])['visit_summary'])
print(ast.literal_eval(df.iloc[index]['problem'])['options'])

B
B
The answer is B: No
visit_summary: 
The patient is a 76-year-old right-handed male who is married and lives with his spouse in a private residence. He is White, speaks English as his primary language, and has completed 16 years of education. He is currently independent in daily living and reports no cognitive concerns, having come to the Alzheimer’s Disease Center primarily to participate in a research study. There is no family history of cognitive impairment among first-degree relatives. The patient is taking three medications, including lisinopril, an ACE inhibitor, and also reports the use of an anxiolytic, sedative, or hypnotic agent, as well as an antihypertensive. He has a history of smoking, having smoked one and a half to two packs per day for 20 years before quitting at age 37. He has no history of traumatic brain injury, stroke, Parkinson’s disease, or other neurological conditions, but he does have active diagnoses of hypertension, diabetes, and thyroid disease.

Physica

In [12]:
generated_texts = list(df['generated_text'])

In [13]:
# generated_text = []
# for gen_text in generated_texts:
#     generated_text += gen_text

In [21]:
generated_text_df = pd.DataFrame(generated_texts, columns=['generated_text']).reset_index(drop=True)

In [22]:
completion_df = pd.DataFrame(generated_text_df['generated_text'].value_counts()).reset_index()

In [23]:
generated_text_df

,generated_text
0,"<think>\nFirst, let's summarize the patient's ..."
1,"<think>\nFirst, let's summarize the patient's ..."
2,"<think>\nFirst, let's provide a comprehensive ..."
3,"<think>\nFirst, let's summarize the patient's ..."
4,"<think>\nFirst, let's summarize the patient's ..."
...,...
20015,"<think>\nFirst, let's summarize the patient's ..."
20016,"<think>\nFirst, let's summarize the patient's ..."
20017,"<think>\nFirst, let's summarize the patient's ..."
20018,"<think>\nFirst, let's summarize the patient's ..."


In [24]:
completion_df[completion_df['count'] > 1]

,generated_text,count
0,"<think>\nFirst, let's summarize the patient's ...",31
1,"<think>\nFirst, let's summarize the patient's ...",13
2,"<think>\nFirst, let's summarize the patient's ...",10
3,"<think>\nFirst, let's provide a comprehensive ...",9
4,"<think>\nFirst, let's summarize the patient's ...",8
5,"<think>\nFirst, summarize the patient's presen...",7
6,"<think>\nFirst, let's summarize the patient's ...",6
7,"<think>\nFirst, let's summarize the patient's ...",6
8,"<think>\nFirst, let's summarize the patient's ...",6
9,"<think>\nFirst, let's summarize the patient's ...",5


In [25]:
completion_df[completion_df['count'] > 1].iloc[51]['generated_text']

"<think>\nFirst, let's summarize the patient's presentation based on the clinical information provided.\n</think>\n<answer>\nAnswer: H\n</answer>"

In [ ]:
from safetensors.torch import load_file
import torch

def safetensors_equal(file1, file2, rtol=1e-5, atol=1e-8):
    tensors1 = load_file(file1)
    tensors2 = load_file(file2)

    if tensors1.keys() != tensors2.keys():
        return False

    for key in tensors1.keys():
        t1 = tensors1[key]
        t2 = tensors2[key]
        
        # If types or shapes don't match, they're not equal
        if t1.dtype != t2.dtype or t1.shape != t2.shape:
            return False
        
        # For floating point, use allclose; for others, use exact match
        if torch.is_floating_point(t1):
            if not torch.allclose(t1, t2, rtol=rtol, atol=atol):
                return False
        else:
            if not torch.equal(t1, t2):
                return False

    return True

# Example usage
equal = safetensors_equal("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/ckpt_sft/qwen25_3B_sft_with_eval/checkpoint-500/model-00001-of-00002.safetensors", "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/ckpt_sft/qwen25_3B_sft_with_eval/model-00001-of-00002.safetensors")
print("Equal:", equal)

In [1]:
def masked_mean(values, mask, axis=None):
    """
    Compute the mean of `values` over elements selected by `mask`.

    Args:
        values (Tensor): Input tensor.
        mask (Tensor): Boolean or numeric mask of the same shape as `values`.
        axis (int or tuple of int, optional): Dimension(s) along which to compute the mean.
            Defaults to None (over all elements).

    Returns:
        Tensor: Masked mean, with shape equal to `values` reduced over `axis`.
    """
    return (values * mask).sum(axis=axis) / (mask.sum(axis=axis) + 1e-8)

In [ ]:
import torch
import torch.nn.functional as F

# Example shapes
batch_size = 2
response_len = 3
vocab_size = 5

# Simulated logits (random for demo purposes)
logits = torch.randn(batch_size, response_len, vocab_size)

# Mask: 1 for real tokens, 0 for padding
mask = torch.tensor([
    [1, 1, 0],  # only 2 valid tokens
    [1, 1, 1]   # all 3 tokens are valid
], dtype=torch.float32)

# ----- Method 1: Entropy = -∑ p log p -----
log_probs = F.log_softmax(logits, dim=-1)
probs = log_probs.exp()
token_entropy_1 = -(probs * log_probs).sum(dim=-1)  # shape: [bs, response_len]

# Masked average entropy per sample
entropy_1 = (token_entropy_1 * mask).sum(dim=-1) / (mask.sum(dim=-1).clamp(min=1.0))

# # ----- Method 2: Entropy = logsumexp - E[z] -----
# softmax_probs = F.softmax(logits, dim=-1)
# token_entropy_2 = torch.logsumexp(logits, dim=-1) - torch.sum(softmax_probs * logits, dim=-1)

# # Masked average entropy per sample
# entropy_21 = ((token_entropy_2 * mask).sum(axis=-1) / (mask.sum(axis=-1) + 1e-8)).mean()
# entropy_22 = masked_mean(token_entropy_2, mask)

# # Compare the two methods
# print("Entropy (method 1):", entropy_1)
# print("Entropy (method 2):", entropy_21, entropy_22)
# # print("Are they close?", torch.allclose(entropy_1, entropy_2, atol=1e-6))


In [10]:
(probs * log_probs) #.sum(dim=-1).shape

tensor([[[-0.2367, -0.3615, -0.3592, -0.2160, -0.1564],
         [-0.3620, -0.0763, -0.3273, -0.2061, -0.3675],
         [-0.2397, -0.2014, -0.1366, -0.2687, -0.2303]],

        [[-0.2364, -0.3392, -0.3406, -0.3677, -0.1439],
         [-0.2221, -0.3368, -0.3008, -0.1419, -0.3565],
         [-0.2951, -0.1989, -0.3608, -0.2750, -0.3071]]])

In [11]:
(probs * log_probs).sum(dim=-1)

tensor([[-1.3298, -1.3392, -1.0768],
        [-1.4278, -1.3581, -1.4370]])

In [12]:
(token_entropy_1 * mask).sum() / mask.sum()

tensor(1.3784)

In [14]:
(token_entropy_1 * mask).sum()

tensor(6.8919)